# 5 量化 Qwen3-1.7B 模型

大语言模型（LLM）在推理部署时面临显存占用高、计算量大、推理时延长等挑战。以 Qwen3-1.7B 为例，其原始权重以 BF16 格式存储，单个参数占用 2 字节，模型权重共需约 3.4GB 显存，对在线体验环境的资源仍有一定要求。

**量化**是工业级大模型部署中降低成本的核心手段之一。其核心思想是将模型中高精度的浮点数（如 BF16/FP32）映射为低精度的整数（如 INT8/INT4），在保留模型推理精度的前提下大幅降低资源消耗。量化的收益主要体现在以下几个方面：

- **降低显存占用**：INT8 仅占 1 字节，相比 BF16 的 2 字节，权重显存占用减半
- **提升计算效率**：INT8 运算的算力密度更高，在昇腾 NPU 上可充分利用 INT8 专用计算单元
- **增加吞吐量**：更小的模型体积和更高的计算效率，使得单位时间内可处理更多请求
- **降低推理时延**：减少数据搬运量和计算时间，单次推理响应更快

**W8A8** 是当前工业界最广泛部署的量化方案，其中 "W" 代表权重（Weight），"A" 代表激活（Activation），"8" 表示 INT8。W8A8 将模型权重和激活值统一量化为 INT8 类型，相比仅量化权重的 W8A16 方案，能同时减少权重存储和激活计算的开销，实现更高的加速比。

本章将以 Qwen3-1.7B 模型为例，使用昇腾量化工具 AMCT（Ascend Model Compression Toolkit）将其 BF16 权重导出为 W8A8 INT8 量化权重，并通过 recipes 推理框架完成量化模型的推理与性能评估，最后从 Profiling 结果中定位耗时占比最高的算子，为下一章的自定义算子开发明确需求来源：

- **环境与工作区准备**：定位仓库、配置运行目录、导入 CANN 环境变量
- **模型权重量化**：使用 AMCT 工具将 Qwen3-1.7B BF16 权重导出为 W8A8 INT8 量化权重
- **量化模型推理**：基于 recipes 框架加载量化权重，验证量化推理功能
- **量化模型 profiling 结果分析**：分析量化模型的算子分布，定位耗时占比最高的算子并归纳其原型定义

---



## 1 环境与工作区准备

准备运行目录，把 CANN 环境变量导入当前 Notebook。


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

print('[环境准备] 开始', flush=True)
os.environ.setdefault('QWEN3_1_7B_DOWNLOAD_BACKEND', 'modelscope')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ.setdefault('ASCEND_RT_VISIBLE_DEVICES', '0')

def locate_repo_root():
    repo_roots = []
    for env_name in ('GITCODE_REPO_ROOT', 'CANN_LEARNING_HUB_ROOT', 'REPO_ROOT'):
        if os.environ.get(env_name):
            repo_roots.append(Path(os.environ[env_name]))
    try:
        cwd = Path.cwd()
        lineage = [cwd, *cwd.parents]
        repo_roots.extend(lineage)
        repo_roots.extend(base / 'cann-learning-hub' for base in lineage)
        for base in lineage:
            try:
                repo_roots.extend(base.glob('*/cann-learning-hub'))
            except OSError:
                pass
    except FileNotFoundError:
        pass
    repo_roots.append(Path('/opt/atomgit/cann-learning-hub'))

    seen = set()
    for repo_root in repo_roots:
        key = str(repo_root)
        if key in seen:
            continue
        seen.add(key)
        if (repo_root / 'tutorials/llm_inference/qwen3_1.7B/src').exists():
            return repo_root
    raise FileNotFoundError('Cannot locate cann-learning-hub repository root. Open this notebook inside cann-learning-hub or set CANN_LEARNING_HUB_ROOT to the repository root.')

REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT / 'tutorials' / 'llm_inference' / 'qwen3_1.7B'
SRC_DIR = TUTORIAL_DIR / 'src'
SOURCE_DIR = REPO_ROOT / 'Sources' / 'model_inference_optimization' / 'qwen3_1.7B'
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
from inference_scripts.environment_check import require_unified_environment
require_unified_environment(TUTORIAL_DIR / 'requirements.txt')

if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME is not set. Please activate the CANN environment before running this notebook.')
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env_cmd = f'source {cann_script} && env'
env = subprocess.check_output(f"bash -lc '{env_cmd}'", shell=True, text=True, cwd=REPO_ROOT)
for line in env.splitlines():
    if '=' in line:
        os.environ.__setitem__(*line.split('=', 1))

RECIPE_ROOT = SRC_DIR / 'inference_scripts' / 'recipe_qwen3_1_7b'
RECIPE_CONFIG_DIR = RECIPE_ROOT / 'models' / 'qwen' / 'config'
A8W8_YAML_TEMPLATE = RECIPE_CONFIG_DIR / 'qwen3_1_7b_a8w8_1tp.yaml'
INFER_SH = RECIPE_ROOT / 'executor' / 'scripts' / 'infer.sh'
MODEL_ID = os.environ.get('QWEN3_1_7B_MODEL_PATH') or 'Qwen/Qwen3-1.7B'

AMCT_DIR = SRC_DIR / 'amct'
AMCT_COMMIT = 'fbd19229eff58e409fea37941db145f76b6ec01e'
W8A8_WEIGHT_DIR = SRC_DIR / 'data' / 'models' / 'Qwen3-1.7B-W8A8'
QUANT_WORK_DIR = SOURCE_DIR / 'run_outputs' / 'quantization_w8a8'
A8W8_YAML = SOURCE_DIR / 'recipe_yaml' / 'qwen3_1_7b_a8w8.yaml'
A8W8_PROFILER_YAML = SOURCE_DIR / 'recipe_yaml' / 'qwen3_1_7b_a8w8_profiler.yaml'
QUANT_WORK_DIR.mkdir(parents=True, exist_ok=True)

print('仓库目录 =', REPO_ROOT)
print('教程目录 =', TUTORIAL_DIR)
print('运行目录 =', SOURCE_DIR)
print('recipes 目录 =', RECIPE_ROOT)
print('推理入口 =', INFER_SH)
print('量化权重目录 =', W8A8_WEIGHT_DIR)
print('可见 NPU =', os.environ.get('ASCEND_RT_VISIBLE_DEVICES'))
print('[环境准备] 完成', flush=True)


## 2 模型权重量化

本节使用昇腾量化工具 [AMCT（Ascend Model Compression Toolkit）](https://gitcode.com/cann/amct) 将已有的 Qwen3-1.7B BF16 权重导出为 W8A8 INT8 量化权重，在下一节用 recipes 框架跑通量化模型推理。


W8A8 INT8 量化将模型权重和激活值从 BF16/FP32 压缩为 INT8，减少内存占用和计算量。昇腾官方大模型量化工具 AMCT 支持通过命令行直接导出 Qwen3-1.7B W8A8 部署权重，输出目录为 HuggingFace 风格的 compressed-tensors 目录结构，recipes 框架可直接加载。

先获取本章需要复用的非量化（BF16）模型权重：


In [ ]:
from inference_scripts.recipe_workflow import resolve_model_path

resolved_model_path, model_source = resolve_model_path(str(MODEL_ID), quiet_model_io=False)
print('非量化模型权重 =', resolved_model_path)
print('模型来源 =', model_source)


使用 AMCT 通过命令行直接导出 Qwen3-1.7B W8A8 INT8 部署权重。流程主要包含如下三步：
1. 获取经过验证的 AMCT 源码版本
2. 复用第 2 章检查完成的统一环境
3. 导出 Qwen3-1.7B W8A8 部署权重

运行下方 code cell 获取 AMCT 源码。本章不会安装 AMCT 仓库中的 `requirements.txt`，避免将统一的 `torch`/`torch_npu` 环境切换为其他版本：


In [ ]:
import subprocess
import sys
from pathlib import Path

print('[AMCT 量化] 开始', flush=True)
if not AMCT_DIR.exists():
    print('[AMCT] 克隆源码...')
    subprocess.check_call(['git', 'clone', 'https://gitcode.com/cann/amct.git', str(AMCT_DIR)])
current_amct_commit = subprocess.check_output(
    ['git', '-C', str(AMCT_DIR), 'rev-parse', 'HEAD'], text=True
).strip()
if current_amct_commit != AMCT_COMMIT:
    print(f'[AMCT] 切换源码版本：{current_amct_commit[:12]} -> {AMCT_COMMIT[:12]}')
    subprocess.check_call(['git', '-C', str(AMCT_DIR), 'checkout', '--detach', AMCT_COMMIT])
print(f'[AMCT] 源码版本 = {AMCT_COMMIT}')


运行下方 code cell 导出 Qwen3-1.7B W8A8 部署权重，其中 `--model` 指定的是非量化模型权重的路径，`--output_dir` 指定的是导出的量化模型权重的路径：

**注意：需要将 `--model` 的参数指定为本地的非量化模型权重的路径**


In [ ]:
# 执行 W8A8 量化导出
print('[AMCT] 执行量化导出...')
deploy_env = os.environ.copy()
deploy_env['PYTHONPATH'] = str(AMCT_DIR)
subprocess.check_call([
    sys.executable, '-m', 'amct_pytorch.deploy',
    '--model', resolved_model_path,
    '--model_name', 'qwen3',
    '--granularity', 'block',
    '--quant_target', 'attn-linear', 'mlp',
    '--quant_dtype', 'int',
    '--bit_config', 'amct_pytorch/configs/w8a8.yaml',
    '--output_dir', str(W8A8_WEIGHT_DIR),
], cwd=str(AMCT_DIR), env=deploy_env)
print(f'[AMCT 量化] 完成，W8A8 权重目录 = {W8A8_WEIGHT_DIR}')

quant_cfg = json.loads((W8A8_WEIGHT_DIR / 'config.json').read_text(encoding='utf-8')).get('quantization_config')
print('quantization_config.quant_method =', (quant_cfg or {}).get('quant_method'))


## 3 量化模型推理


### 3.1 复用统一环境

量化推理直接复用第 2 章检查完成的统一环境，本章不再执行依赖安装。


### 3.2 准备 W8A8 推理 YAML

W8A8 量化推理使用与 BF16 推理相同的 recipes 框架，但需要使用 W8A8 量化模型路径和适配的 YAML 配置。量化配置由权重目录 `config.json` 内嵌的 `quantization_config` 自动加载，YAML 无需显式声明量化参数。

下方 code cell 从模板 `qwen3_1_7b_a8w8_1tp.yaml` 生成运行 YAML，并把其中的模型路径指定为量化权重路径 `W8A8_WEIGHT_DIR`：

**注意：需要将 `W8A8_WEIGHT_DIR` 指定为本地的量化模型权重的路径**


In [ ]:
import os
import time
from pathlib import Path

import yaml
from inference_scripts.recipe_workflow import (
    collect_recipe_metrics,
    find_recipe_case_dir,
    prepare_runtime_yaml,
    prepare_profiling_yaml,
    print_yaml_focus,
)
from inference_scripts.inference_qwen3_1_7b import ensure_recipe_log_ok

def status_cn(status):
    return {'ok': '通过', 'warning': '告警', 'failed': '失败'}.get(status, status)

def load_metrics(work_dir, tag):
    metrics_path = Path(work_dir) / 'metrics' / f'{tag}_summary.json'
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('指标文件 =', metrics_path)
    print('recipes YAML =', metrics.get('recipe_yaml_path'))
    print('recipes 结果目录 =', metrics.get('recipe_case_dir'))
    print('生成文件 =', metrics.get('outputs_path'))
    print('生成 token 数 =', metrics.get('generated_tokens'))
    print('总耗时(s) =', metrics.get('total_time_s'))
    print('吞吐(tokens/s) =', metrics.get('tokens_per_second'))
    print('profiler 开启 =', metrics.get('profiler_enabled'))
    sanity = metrics.get('generation_text_sanity')
    if sanity:
        print('吐字检查 =', status_cn(sanity.get('status')), f"警告={sanity.get('num_with_warnings')}", f"失败={sanity.get('num_failed')}")
    return metrics

def set_run_env(yaml_path, work_dir, log_name):
    log_dir = Path(work_dir) / 'recipe_logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    os.environ['QWEN3_RECIPE_ROOT'] = str(RECIPE_ROOT)
    os.environ['QWEN3_RUN_YAML'] = str(yaml_path)
    os.environ['QWEN3_RUN_LOG'] = str(log_dir / log_name)
    os.environ['QWEN3_RUN_START_TIME'] = str(time.perf_counter())
    os.environ['QWEN3_SKIP_RECIPE_RUN'] = '0'

def prepare_quant_run(work_dir, yaml_path, template_path, log_name):
    prepare_runtime_yaml(Path(template_path), Path(yaml_path), model_path=str(W8A8_WEIGHT_DIR))
    print('参考 YAML =', template_path)
    print('运行 YAML =', yaml_path)
    print_yaml_focus(Path(yaml_path))
    set_run_env(yaml_path, work_dir, log_name)

def collect_quant_run(work_dir, metrics_tag, yaml_path, optimization_mode):
    case_dir = find_recipe_case_dir(RECIPE_ROOT, Path(yaml_path))
    if case_dir is None:
        raise RuntimeError(f'未找到 recipes 结果目录：{Path(yaml_path).stem}')
    recipe_log_path = case_dir / 'log_0.log'
    ensure_recipe_log_ok(recipe_log_path)
    config = yaml.safe_load(Path(yaml_path).read_text(encoding='utf-8'))
    scheduler_config = config.get('scheduler_config', {})
    data_config = config.get('data_config', {})
    run_info = {
        'stdout_path': os.environ['QWEN3_RUN_LOG'],
        'stderr_path': os.environ['QWEN3_RUN_LOG'],
        'wall_time_s': time.perf_counter() - float(os.environ['QWEN3_RUN_START_TIME']),
        'recipe_case_dir': str(case_dir),
        'recipe_log_path': str(recipe_log_path),
    }
    return collect_recipe_metrics(
        RECIPE_ROOT,
        Path(work_dir),
        metrics_tag,
        Path(yaml_path),
        run_info,
        model_id=str(MODEL_ID),
        resolved_model_path=str(W8A8_WEIGHT_DIR),
        model_source='amct_export',
        optimization_mode=optimization_mode,
        max_new_tokens=int(scheduler_config.get('max_new_tokens', 64)),
        max_input_tokens=int(data_config.get('input_truncated_len', 256)),
        enable_thinking=bool(scheduler_config.get('enable_thinking', False)),
    )

print('[YAML 准备] W8A8 量化推理', flush=True)
prepare_quant_run(QUANT_WORK_DIR, A8W8_YAML, A8W8_YAML_TEMPLATE, 'quant_infer_stdout.log')


### 3.3 执行量化推理

调用 `infer.sh` 执行量化推理，查看 W8A8 量化模型推理生成的文本，确认量化推理正常运行。


In [ ]:
%%bash
set -euo pipefail

if [ "${QWEN3_SKIP_RECIPE_RUN:-0}" = "1" ]; then
  echo '跳过本次 recipes 推理。'
  exit 0
fi

echo 'recipes 启动命令:'
echo "cd ${QWEN3_RECIPE_ROOT}"
echo "bash executor/scripts/infer.sh --model qwen --mode offline --yaml ${QWEN3_RUN_YAML}"
cd "${QWEN3_RECIPE_ROOT}"
bash executor/scripts/infer.sh --model qwen --mode offline --yaml "${QWEN3_RUN_YAML}" 2>&1 | tee "${QWEN3_RUN_LOG}"


In [ ]:
quant_metrics = collect_quant_run(QUANT_WORK_DIR, 'quant_w8a8_infer', A8W8_YAML, 'quant_w8a8')
quant_metrics = load_metrics(QUANT_WORK_DIR, 'quant_w8a8_infer')
generated_text = Path(quant_metrics['outputs_path']).read_text(encoding='utf-8').splitlines()
if generated_text:
    print('生成文本示例 =', json.loads(generated_text[0]).get('generated_text', '')[:120])


## 4 量化模型 profiling 结果分析

本节使用 profiling 工具观测量化模型的推理性能。完成带 Profiler 的推理后，会在结果目录中生成 `prof/` 产物。

推理完成后，定位 `prof/` 目录，重点关注其中的 `op_statistic.csv` 和 `kernel_details.csv` 这两个文件。`op_statistic.csv` 中展示了模型调用的所有算子的统计信息，`kernel_details.csv` 中展示了详细的接口调用信息。

首先基于 `op_statistic.csv` 分析量化 Qwen3-1.7B 模型中耗时占比最高的算子，定位性能瓶颈；然后进一步根据 `kernel_details.csv` 分析该算子的输入输出规格，归纳其调用模式，最终推导出自定义算子 `QmmCustom` 的原型定义，从而明确下一章的自定义算子开发的需求来源。

### 4.1 进行 Profiling 推理

沿用 recipes 的 YAML + `infer.sh` 工作流，在配置中打开 Profiler，运行一次推理并收集性能数据。

通过如下操作将量化模型 YAML 配置中的 `enable_profiler` 配置为 `True`，并启动带 Profiler 的推理：


In [ ]:
print('[YAML 准备] W8A8 量化推理 + Profiler', flush=True)
prepare_profiling_yaml(A8W8_YAML, A8W8_PROFILER_YAML, enable_profiler=True)
print_yaml_focus(A8W8_PROFILER_YAML)
set_run_env(A8W8_PROFILER_YAML, QUANT_WORK_DIR, 'quant_infer_profiler_stdout.log')


In [ ]:
%%bash
set -euo pipefail

if [ "${QWEN3_SKIP_RECIPE_RUN:-0}" = "1" ]; then
  echo '跳过本次 recipes 推理。'
  exit 0
fi

echo 'recipes 启动命令:'
echo "cd ${QWEN3_RECIPE_ROOT}"
echo "bash executor/scripts/infer.sh --model qwen --mode offline --yaml ${QWEN3_RUN_YAML}"
cd "${QWEN3_RECIPE_ROOT}"
bash executor/scripts/infer.sh --model qwen --mode offline --yaml "${QWEN3_RUN_YAML}" 2>&1 | tee "${QWEN3_RUN_LOG}"


In [ ]:
quant_profiler_metrics = collect_quant_run(QUANT_WORK_DIR, 'quant_w8a8_profiler', A8W8_PROFILER_YAML, 'quant_w8a8_profiler')
quant_profiler_metrics = load_metrics(QUANT_WORK_DIR, 'quant_w8a8_profiler')
print('Profiler 摘要 =', quant_profiler_metrics.get('profiler_summary_path'))


### 4.2 基于 op_statistic.csv 分析算子调用

`op_statistic.csv` 是昇腾 Profiler 生成的算子级统计汇总文件，记录了推理过程中每个算子的调用次数、总耗时、最小/平均/最大耗时及耗时占比（Ratio）。通过分析该文件，可以快速定位模型推理的性能瓶颈——即哪些算子消耗了最多的计算时间。

对于量化模型而言，由于权重和激活值从 BF16 压缩为 INT8，矩阵乘法类算子的计算模式发生了变化：原本的 BF16 MatMul 被替换为 INT8 量化矩阵乘法算子 `QuantBatchMatmulV3`。通过 `op_statistic.csv` 分析量化模型的算子分布，可以直观了解量化后的算子调用特征，并识别出耗时占比最高的关键算子，为后续的自定义算子优化提供方向。

下方代码分别读取 Prefill 和 Decode 阶段的 `op_statistic.csv`，展示按耗时占比排序的前十类算子：


In [ ]:
import pandas as pd
from pathlib import Path

case_dir = find_recipe_case_dir(RECIPE_ROOT, A8W8_PROFILER_YAML)
assert case_dir is not None, '未找到 profiling 结果目录，请确认 4.1 的 Profiling 推理已完成'
prof_dir = Path(case_dir) / 'prof'
print('结果目录 =', case_dir)

def find_phase_csv(phase, name):
    matches = sorted((prof_dir / phase).rglob(name), key=lambda p: p.parent.parent.name, reverse=True)
    return matches[0] if matches else None

for phase in ('prefill', 'decode'):
    csv_path = find_phase_csv(phase, 'op_statistic.csv')
    if not csv_path:
        print(f'{phase} op_statistic.csv 未找到')
        continue
    df = pd.read_csv(csv_path)
    if 'Ratio(%)' in df.columns:
        df = df.sort_values('Ratio(%)', ascending=False)
    print(f'{phase} op_statistic.csv =', csv_path)
    display(df.head(10))


从 `op_statistic.csv` 中统计的算子调用信息可见，`QuantBatchMatmulV3`（CANN 内置的量化 matmul 算子）是整个模型推理过程中调用耗时占比最高的算子。


### 4.3 基于 kernel_details.csv 分析算子原型

在 4.2 节中，我们通过 `op_statistic.csv` 已确认 `QuantBatchMatmulV3` 是量化模型中耗时占比最高的算子。然而 `op_statistic.csv` 仅提供算子级别的宏观统计（调用次数、总耗时），缺少具体的输入输出规格。要理解该算子的完整行为，我们需要从 `kernel_details.csv` 中提取每次调用的详细参数——包括输入输出的 Shape、Data Type 和 Format。

这些信息至关重要：它们构成了算子的“原型定义”，即算子需要接收什么、输出什么。正如软件开发中需要先明确接口规格再进行编码，自定义算子的开发同样需要从实际调用数据中归纳出完整的原型定义，才能确保开发出的算子能无缝接入模型推理流程。

下方代码筛选 `QuantBatchMatmulV3` 的所有调用记录，按 Input Shapes 分组统计其在 Prefill 和 Decode 阶段的输入输出规格与平均耗时：


In [ ]:
import pandas as pd
from pathlib import Path

case_dir = find_recipe_case_dir(RECIPE_ROOT, A8W8_PROFILER_YAML)
assert case_dir is not None, '未找到 profiling 结果目录，请确认 4.1 的 Profiling 推理已完成'
prof_dir = Path(case_dir) / 'prof'

def find_kernel_csv(phase):
    matches = sorted((prof_dir / phase).rglob('kernel_details.csv'), key=lambda p: p.parent.parent.name, reverse=True)
    return matches[0] if matches else None

prefill_csv = find_kernel_csv('prefill')
decode_csv = find_kernel_csv('decode')
assert prefill_csv and decode_csv, 'kernel_details.csv 未找到，请确认 profiling 已完成'
print(f'Prefill CSV: {prefill_csv}')
print(f'Decode CSV: {decode_csv}')

group_keys = ['Input Shapes', 'Input Data Types', 'Input Formats', 'Output Shapes', 'Output Data Types']

def qmm_duration_stats(df, phase):
    df = df[df['Type'] == 'QuantBatchMatmulV3'].copy()
    present = [k for k in group_keys if k in df.columns]
    stats = df.groupby(present)['Duration(us)'].agg(['mean', 'count']).reset_index()
    stats = stats.rename(columns={'mean': 'Avg Duration(us)', 'count': 'Calls'})
    stats = stats.sort_values('Avg Duration(us)', ascending=False).reset_index(drop=True)
    stats.insert(0, 'Phase', phase)
    return stats

prefill_stats = qmm_duration_stats(pd.read_csv(prefill_csv), 'Prefill')
decode_stats = qmm_duration_stats(pd.read_csv(decode_csv), 'Decode')
combined = pd.concat([prefill_stats, decode_stats], ignore_index=True)
display(combined.style.set_caption('QuantBatchMatmulV3 算子调用情况'))


从上表的 `QuantBatchMatmulV3` 调用数据中，我们可以归纳出该算子的核心行为模式：

- **输入始终包含三个核心张量**：`x1`（INT8, ND 格式）为激活矩阵 A，`x2`（INT8, **FRACTAL_NZ** 格式）为权重矩阵 B，`scale`（FLOAT32, ND 格式）为 per-channel 缩放系数。其中 FRACTAL_NZ 是昇腾 NPU 针对 INT8 矩阵乘法优化的特殊存储格式，将权重按 16×32 分块重排，以充分利用 AI Core 的 Cube 计算单元。

- **pertoken_scale 决定输出类型**：部分调用模式携带第 4 个输入 `pertoken_scale`（FLOAT32, ND 格式），此时输出为 BF16——INT32 中间结果经 perTokenScale 反量化回低精度浮点；不含该输入时输出为 INT32，保留整型结果供后续算子继续处理。

- **Shape 随推理阶段和模型层级变化**：Prefill 阶段 M 维度较大，Decode 阶段 M=1；K 和 N 取值取决于具体层的隐藏维度与投影维度。

现在让我们模拟一个实际的需求分析场景：假设我们正在部署一个 W8A8 量化的大语言模型，推理过程中需要调用 INT8 量化矩阵乘法算子。通过 Profiling 分析，我们发现该模型对底层算子的调用规格恰好如上表所示——即需要一种能同时支持“INT32 输出”和“perTokenScale 反量化输出”两种模式的量化 MatMul 算子。

然而，现有框架中可能没有完全匹配该规格的算子实现，或者我们需要针对特定硬件平台进行深度优化。此时，就需要开发一个自定义算子 `QmmCustom` 来满足这一需求。从 Profiling 数据中归纳出的输入输出规格，自然就成为 `QmmCustom` 的原型定义：

| 算子名称 | QmmCustom |
|------|-----|
| 输入 x1 (A) | INT8, ND 格式, shape [M, K] |
| 输入 x2 (B) | INT8, **FRACTAL_NZ** 格式, shape [K, N] |
| 输入 scale | FLOAT32, ND 格式, shape [N] |
| 输入 pertoken_scale | FLOAT32, ND 格式, shape [M]（可选） |
| 输出 y | 有 perTokenScale → BF16，无 → INT32；ND 格式, shape [M, N] |
| 架构 | ascend910b |


---

### 小结

本章完成了 Qwen3-1.7B 模型的权重量化，通过 AMCT 工具将 BF16 权重导出为 W8A8 INT8 量化权重，并验证了量化模型的推理功能。随后通过 Profiling 分析定位到耗时占比最高的算子——量化 matmul（`QuantBatchMatmulV3`），并从 `kernel_details.csv` 中归纳出该算子的输入输出规格，推导出自定义算子 `QmmCustom` 的原型定义。

在下一章中，我们将基于本章归纳的算子原型，完成 `QmmCustom` 算子的 Tiling 设计、Kernel 实现、编译与测试，并将其接入 Qwen3-1.7B 量化模型，验证自定义算子在模型推理中的功能正确性与性能表现。
